# The agentic tool-call loop

The agentic loop is the foundation behind all agent frameworks. It's also called the tool call loop or function call loop.

In [4]:
from openai import OpenAI

openai_client = OpenAI()

In [5]:
from gitsource import GithubRepositoryDataReader, chunk_documents

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

In [6]:
import json

from minsearch import AppendableIndex

index = AppendableIndex(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)

def search(query):
    results = index.search(
        query=query,
        num_results=5
    )
    return results

def make_call(tool_call):
    arguments = json.loads(tool_call.arguments)
    name = tool_call.name

    if name == 'search':
        result = search(**arguments)
    else: 
        result = 'not found tool "{name}"'
    
    return {
        "type": "function_call_output",
        "call_id": tool_call.call_id,
        "output": json.dumps(result),
    }


In [8]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the documentation database for relevant results based on a query string.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query to look up in the index"
            }
        },
        "required": [
            "query"
        ]
    }
}


In [9]:
instructions = """
You're a documentation assistant. 

Answer the user question using the documentation knowledge base

Make 3 iterations:

1) in the first iteration, perform one search
2) in the second interation, analyze the results from the previous search
   and perform 2 more searches
3) synthesise the results into the output

IMPORTANT: at each step, give an explanation of why you want to perform 
search for this particular search query. It should be 2-3 sentences explaining
the logic of your decision.

Use only facts from the knowledge base when answering.
If you cannot find the answer, inform the user.

Our knowledge base is entirely about Evidently, so you don't need to 
include the word 'evidently' in search results
"""

question = "How do I create a dahsbord in Evidently?"


message_history = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]

iteration_number = 1

while True:
    response = openai_client.responses.create(
        model='gpt-4o-mini',
        input=message_history,
        tools=[search_tool],
    )

    print(f'iteraration number {iteration_number}...') 
    message_history.extend(response.output)

    has_function_calls = False

    for message in response.output:
        if message.type == 'function_call':
            print(f'executing {message.name}({message.arguments})...')
            tool_call_output = make_call(message)
            message_history.append(tool_call_output)
            has_function_calls = True

        if message.type == 'message':
            text = message.content[0].text
            print('ASSISTANT:', text)

    iteration_number = iteration_number + 1
    print()
    
    if not has_function_calls:
        break


iteraration number 1...
ASSISTANT: To start, I will search for information specifically about creating a dashboard. This will help me find instructions or guidelines that directly address the process involved in setting up a dashboard, ensuring I provide accurate and relevant details.

Let's proceed with the search.
executing search({"query":"create dashboard"})...

iteraration number 2...
ASSISTANT: I've found relevant information on creating a dashboard. You can add panels, which visualizes evaluation results over time, and you can organize these panels into tabs. It's crucial to add reports with evaluation results to your project first, as this enables the panels to display data correctly. 

Now, I will analyze these results to provide a comprehensive overview and identify additional specific steps or options available. For this, I will perform two more searches: one on customizing panels and another focusing on dashboard management. This way, I can ensure you have detailed and usef

In [12]:
def add_entry(filename, title, description, content):
    entry = {
        'start': 0,
        'content': content,
        'title': title,
        'description': description,
        'filename': filename,
    }
    index.append(entry)
    return "OK"

add_entry_tool = {
    "type": "function",
    "name": "add_entry",
    "description": "Add a new documentation entry to the index.",
    "parameters": {
        "type": "object",
        "properties": {
            "filename": {
                "type": "string",
                "description": "The source filename associated with the entry"
            },
            "title": {
                "type": "string",
                "description": "The title of the documentation entry"
            },
            "description": {
                "type": "string",
                "description": "A short description summarizing the entry"
            },
            "content": {
                "type": "string",
                "description": "The full content of the documentation entry"
            }
        },
        "required": [
            "filename",
            "title",
            "description",
            "content"
        ]
    }
}

In [10]:
def make_call(tool_call):
    arguments = json.loads(tool_call.arguments)
    name = tool_call.name

    if name == 'search':
        result = search(**arguments)
    elif name == 'add_entry':
        result = add_entry(**arguments)
    else: 
        result = 'not found tool "{name}"'
    
    return {
        "type": "function_call_output",
        "call_id": tool_call.call_id,
        "output": json.dumps(result),
    }

In [13]:
message_history = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]
tools = [search_tool, add_entry_tool]

iteration_number = 1

while True:
    response = openai_client.responses.create(
        model='gpt-4o-mini',
        input=message_history,
        tools=tools,
    )

    print(f'iteraration number {iteration_number}...') 
    message_history.extend(response.output)

    has_function_calls = False

    for message in response.output:
        if message.type == 'function_call':
            print(f'executing {message.name}({message.arguments})...')
            tool_call_output = make_call(message)
            message_history.append(tool_call_output)
            has_function_calls = True

        if message.type == 'message':
            text = message.content[0].text
            print('ASSISTANT:', text)

    iteration_number = iteration_number + 1
    print()
    
    if not has_function_calls:
        break


iteraration number 1...
ASSISTANT: To assist with your query about creating a dashboard, I'll start by searching for documentation related to dashboard creation. This will help identify the specific steps and guidelines you need to follow to successfully create a dashboard.

Let's begin the search.
executing search({"query":"create dashboard"})...

iteraration number 2...
ASSISTANT: From the initial search, I've found detailed information on creating a dashboard and adding panels in the platform. Key steps include:

1. **Adding Tabs:** Enter "Edit" mode and use the "+" button to create new tabs which help organize your panels.
2. **Adding Panels:** In "Edit" mode, click "Add Panel" to configure various panel types such as text, counters, or plots. Ensure you have metrics logged in reports for the panels to display correctly.

Next, I'll analyze these instructions more closely and expand the search to find additional documentation on the types of panels available and any specific config

In [14]:
message_history

[{'role': 'system',
  'content': "\nYou're a documentation assistant. \n\nAnswer the user question using the documentation knowledge base\n\nMake 3 iterations:\n\n1) in the first iteration, perform one search\n2) in the second interation, analyze the results from the previous search\n   and perform 2 more searches\n3) synthesise the results into the output\n\nIMPORTANT: at each step, give an explanation of why you want to perform \nsearch for this particular search query. It should be 2-3 sentences explaining\nthe logic of your decision.\n\nUse only facts from the knowledge base when answering.\nIf you cannot find the answer, inform the user.\n\nOur knowledge base is entirely about Evidently, so you don't need to \ninclude the word 'evidently' in search results\n"},
 {'role': 'user', 'content': 'How do I create a dahsbord in Evidently?'},
 ResponseOutputMessage(id='msg_03fec19bd01550340069f49f3b06988191948af6adf427d7a6', content=[ResponseOutputText(annotations=[], text="To assist with 

In [15]:
message_history.append(
    {"role": "user", "content": "add this content to our database"}
)

In [16]:
while True:
    response = openai_client.responses.create(
        model='gpt-4o-mini',
        input=message_history,
        tools=tools,
    )

    print(f'iteraration number {iteration_number}...') 
    message_history.extend(response.output)

    has_function_calls = False

    for message in response.output:
        if message.type == 'function_call':
            print(f'executing {message.name}({message.arguments})...')
            tool_call_output = make_call(message)
            message_history.append(tool_call_output)
            has_function_calls = True

        if message.type == 'message':
            text = message.content[0].text
            print('ASSISTANT:', text)

    iteration_number = iteration_number + 1
    print()
    
    if not has_function_calls:
        break


iteraration number 4...
executing add_entry({"filename":"docs/platform/create_dashboard_guide.mdx","title":"Creating Dashboards","description":"Guide on how to create and customize dashboards with tabs and panels.","content":"### Steps to Create a Dashboard\n\n1. **Access Dashboard Management:**\n   - Make sure you're connected to the required environment and have created a project.\n\n2. **Creating Tabs:**\n   - Enter **Edit Mode** on the dashboard (top right corner).\n   - Click the \" + \" sign to add a new tab, which helps organize panels.\n   - You can choose to create a custom tab or start with pre-built templates that come with preset panel combinations.\n\n3. **Adding Panels:**\n   - With the dashboard in edit mode, click \"Add Panel\" to bring up the panel configuration interface.\n   - Panels can include the following types:\n     - **Text Panels**: Ideal for titles or descriptions.\n     - **Counter Panels**: Display values, such as totals or averages.\n     - **Pie Charts, 

In [17]:
index.docs[-1]

{'start': 0,
 'content': '### Steps to Create a Dashboard\n\n1. **Access Dashboard Management:**\n   - Make sure you\'re connected to the required environment and have created a project.\n\n2. **Creating Tabs:**\n   - Enter **Edit Mode** on the dashboard (top right corner).\n   - Click the " + " sign to add a new tab, which helps organize panels.\n   - You can choose to create a custom tab or start with pre-built templates that come with preset panel combinations.\n\n3. **Adding Panels:**\n   - With the dashboard in edit mode, click "Add Panel" to bring up the panel configuration interface.\n   - Panels can include the following types:\n     - **Text Panels**: Ideal for titles or descriptions.\n     - **Counter Panels**: Display values, such as totals or averages.\n     - **Pie Charts, Line Plots, Bar Plots**: Visualize trends and distributions over time.\n   - Specify metrics to be displayed on each panel and set their properties as needed, such as size ("full" or "half") and plot typ

## The Q&A loop

In [18]:
tools = [search_tool, add_entry_tool]

message_history = [
    {"role": "system", "content": instructions},
]

iteration_number = 1

# Q&A loop
while True:
    user_prompt = input('You:')
    if user_prompt.lower().strip() == 'stop':
        break

    message_history.append({"role": "user", "content": user_prompt})

    # tool-call loop
    while True:
        response = openai_client.responses.create(
            model='gpt-4o-mini',
            input=message_history,
            tools=tools,
        )
    
        print(f'iteraration number {iteration_number}...') 
        message_history.extend(response.output)
    
        has_function_calls = False
    
        for message in response.output:
            if message.type == 'function_call':
                print(f'executing {message.name}({message.arguments})...')
                tool_call_output = make_call(message)
                message_history.append(tool_call_output)
                has_function_calls = True
    
            if message.type == 'message':
                text = message.content[0].text
                print('ASSISTANT:', text)
    
        iteration_number = iteration_number + 1
        print()
        
        if not has_function_calls:
            break


iteraration number 1...
executing search({"query":"llm as judge"})...

iteraration number 2...
executing search({"query":"creating LLM judge"})...
executing search({"query":"LLM evaluator tutorial"})...

iteraration number 3...
ASSISTANT: ### LLM as Judge

The concept of utilizing a Large Language Model (LLM) as a judge involves employing the model to evaluate text according to specific criteria, such as correctness, style, and verbosity. This process includes two approaches: reference-based and open-ended evaluations.

1. **Reference-Based Evaluation**: This approach requires comparing new responses against approved or "ground truth" answers. It's particularly useful for regression testing, where the goal is to ensure that a model still provides accurate responses after changes are made.

2. **Open-Ended Evaluation**: This method assesses new outputs based on custom-defined criteria when no reference is available, allowing for more flexible evaluation.

### Key Steps to Create and Run